### Installing Required Libraries

In this step, I installed all the required libraries needed to build the Advanced RAG pipeline.

- `sentence-transformers` is used for generating dense embeddings (SBERT).
- `rank-bm25` is used for keyword-based retrieval.
- `google-generativeai` is used to access Gemini for query expansion and answer generation.
- `groq` is included for optional fast LLM usage.
- `langchain` helps in structuring LLM-based pipelines.

In [14]:
!pip install -q sentence-transformers rank-bm25 groq google-generativeai langchain langchain-community langchain-google-genai

### Loading API Keys

In this step, I securely load API keys for Gemini and Groq.

In [15]:
import os
import getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

print("Keys loaded successfully")

Keys loaded successfully


### Creating the Document Corpus

In this step, I construct the document corpus which serves as the knowledge base for the RAG system.

Corpus Details:
- Total number of documents: 12  
- Each document contains: 2 sentences  
- Domain: Artificial Intelligence and Machine Learning  

Design Choices:

1. Richer Context:
   Each document contains two sentences instead of one. This provides more contextual information, which improves semantic understanding during embedding and re-ranking.

2. Topic Coverage:
   The corpus includes a variety of AI/ML topics such as:
   - Transformers and attention mechanisms
   - Optimization techniques (Gradient Descent, Adam)
   - Neural network training (Backpropagation)
   - Retrieval methods (BM25, Sentence Transformers, Cross-Encoders)
   - Model evaluation (Overfitting, Regularization)

3. Related Sub-topics:
   Multiple documents focus on similar areas to test retrieval quality:
   - Training and optimization (Gradient Descent, Backpropagation, Adam)
   - Transformer-based concepts (Transformers, Attention)

4. Keyword-heavy Content:
   Some documents include technical terms such as "BM25" and "Cross-Encoder". These help keyword-based retrieval methods like BM25 perform effectively.

5. Balanced Design:
   The corpus is designed to support both:
   - Lexical matching (BM25)
   - Semantic similarity (SBERT)


In [16]:
corpus = [
    "Transformers use self-attention to process sequences in parallel. This allows them to capture long-range dependencies more effectively than RNNs.",

    "BERT is a bidirectional encoder trained using masked language modeling. It understands context from both left and right of a word.",

    "Gradient descent minimizes loss using iterative updates. Variants like stochastic gradient descent improve training efficiency.",

    "Backpropagation computes gradients using the chain rule. It is essential for training neural networks by updating weights.",

    "Adam optimizer combines momentum and adaptive learning rates. It is widely used for faster convergence in deep learning models.",

    "Attention mechanisms allow models to focus on relevant tokens. This improves performance in tasks like translation and summarization.",

    "Tokenization splits text into smaller units like words or subwords. It is a crucial preprocessing step in NLP pipelines.",

    "The BM25 algorithm ranks documents using term frequency and inverse document frequency. It is effective for keyword-based retrieval tasks.",

    "Sentence Transformers generate dense vector embeddings for semantic similarity. These embeddings are used in modern retrieval systems.",

    "Cross-encoders jointly encode query and document for relevance scoring. They provide more accurate results but are computationally expensive.",

    "Overfitting occurs when a model memorizes training data instead of generalizing. This leads to poor performance on unseen data.",

    "Regularization techniques like dropout reduce overfitting. They improve model generalization by preventing reliance on specific neurons."
]

### Implementing Hybrid Retrieval (BM25 + SBERT + RRF)

In this step, I implemented a hybrid retrieval system.

It combines:
- BM25 for keyword-based retrieval
- SBERT for semantic similarity

To combine both rankings, I used Reciprocal Rank Fusion (RRF). This method assigns higher scores to documents that rank well in both systems.

The retriever returns:
- BM25 rank
- SBERT rank
- RRF score
- Document text

In [17]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np

class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)

        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.embeddings = self.model.encode(corpus, convert_to_numpy=True)

    def retrieve(self, query, top_k=8):
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        q_emb = self.model.encode([query])[0]
        sims = np.dot(self.embeddings, q_emb) / (
            np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(q_emb)
        )
        sbert_ranks = np.argsort(sims)[::-1]

        results = []

        for i in range(len(self.corpus)):
            bm_rank = np.where(bm25_ranks == i)[0][0]
            sb_rank = np.where(sbert_ranks == i)[0][0]

            rrf = 1/(self.k + bm_rank) + 1/(self.k + sb_rank)

            results.append({
                "doc_id": i,
                "text": self.corpus[i],
                "bm25_rank": int(bm_rank),
                "sbert_rank": int(sb_rank),
                "rrf_score": rrf
            })

        return sorted(results, key=lambda x: x["rrf_score"], reverse=True)[:top_k]

### Cross-Encoder Re-Ranking

In this step, I usde a cross-encoder to re-rank retrieved documents.

Unlike SBERT, which encodes query and documents separately, the cross-encoder processes them together. This improves accuracy in relevance scoring.

Although it is slower, it significantly improves the quality of final retrieved documents.

In [18]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for doc, score in zip(candidates, scores):
        doc["cross_score"] = float(score)

    return sorted(candidates, key=lambda x: x["cross_score"], reverse=True)[:top_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Query Expansion using Multi-Query

In this step, I improved retrieval by expanding the user query.

I used Gemini to generate multiple paraphrases of the query. This helps:
- Capture different meanings
- Improve recall
- Handle vague or short queries

All generated queries are used for retrieval.

In [19]:
import google.generativeai as genai

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash")

def generate_queries(query):
    prompt = f"""
Generate exactly 3 short paraphrases of this query.
Do not number them.

Query: {query}
"""
    response = model.generate_content(prompt)
    variants = [v.strip() for v in response.text.split("\n") if v.strip()]
    return [query] + variants[:3]

### Building the Advanced RAG Pipeline

In this step, I integrated all components into a single pipeline.

Steps:
1. Expand the query using Gemini
2. Retrieve documents using hybrid retrieval
3. Merge and remove duplicate documents
4. Re-rank using cross-encoder
5. Generate final answer using Gemini

This pipeline significantly improves answer quality compared to naive RAG.

In [20]:
retriever = HybridRetriever(corpus)

def advanced_rag(user_query):
    queries = generate_queries(user_query)

    all_docs = []
    seen = set()

    for q in queries:
        results = retriever.retrieve(q, top_k=5)
        for doc in results:
            if doc["text"] not in seen:
                seen.add(doc["text"])
                all_docs.append(doc)

    reranked = rerank(user_query, all_docs, top_k=3)

    context = "\n".join([doc["text"] for doc in reranked])

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{user_query}
"""

    response = model.generate_content(prompt)
    return response.text, reranked

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Implementing Naive RAG

This is a baseline system that uses only SBERT similarity for retrieval.

It does not include:
- Query expansion
- Hybrid retrieval
- Re-ranking

This helps demonstrate the improvement achieved by the advanced pipeline.

In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np

naive_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embs = naive_model.encode(corpus)

def naive_rag(query):
    q_emb = naive_model.encode([query])[0]
    sims = np.dot(doc_embs, q_emb)
    idx = np.argmax(sims)
    return corpus[idx]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Comparing Naive RAG and Advanced RAG

In this step, I evaluated both systems on the same queries.

For each query, I compared:
- The top document retrieved by naive RAG
- The top document retrieved by advanced RAG
- Whether the results differ

This demonstrates the effectiveness of hybrid retrieval and re-ranking.

In [22]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is attention mechanism"
]

print("| Query | Naive Top Doc | Advanced Top Doc | Different? |")
print("|------|----------------|------------------|------------|")

for q in queries:
    naive_doc = naive_rag(q)
    adv_answer, adv_docs = advanced_rag(q)
    adv_doc = adv_docs[0]["text"]

    diff = "Yes" if naive_doc != adv_doc else "No"

    print(f"| {q} | {naive_doc} | {adv_doc} | {diff} |")

| Query | Naive Top Doc | Advanced Top Doc | Different? |
|------|----------------|------------------|------------|
| how do transformers encode meaning? | Transformers use self-attention to process sequences in parallel. This allows them to capture long-range dependencies more effectively than RNNs. | Sentence Transformers generate dense vector embeddings for semantic similarity. These embeddings are used in modern retrieval systems. | Yes |
| optimization techniques for training | Gradient descent minimizes loss using iterative updates. Variants like stochastic gradient descent improve training efficiency. | Backpropagation computes gradients using the chain rule. It is essential for training neural networks by updating weights. | Yes |
| what is attention mechanism | Attention mechanisms allow models to focus on relevant tokens. This improves performance in tasks like translation and summarization. | Attention mechanisms allow models to focus on relevant tokens. This improves perfor